# Diagnostico da alocacao

Notebook enxuto para testar a viabilidade da alocacao por dia.

Mantem apenas:

- `testar_use_soft_por_dia`
- `comparar_restricoes_incrementais`
- dependencias necessarias para montar e resolver o modelo.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from ortools.sat.python import cp_model

PROJECT_ROOT = Path('/home/cayena/Projeto_thefamily/TeacherScheduler')
sys.path.append(str(PROJECT_ROOT))

from streamlit_app.teacher_alocation import TeacherScheduler
from streamlit_app.utils import transform_classes_dateframe, transform_teacher_dataframe

## 1. Carregar dados

Ajuste os caminhos abaixo quando quiser testar outra rota ou outra base de professores.

In [ ]:
CAMINHO_AULAS = Path('/home/cayena/Downloads/rota ARRUMADA 2.xlsx')
CAMINHO_PROFESSORES = Path('/home/cayena/Downloads/professor 5.xlsx')

aulas_raw = pd.read_excel(CAMINHO_AULAS)
professores_raw = pd.read_excel(CAMINHO_PROFESSORES)

df_result = transform_classes_dateframe(aulas_raw)
df_professor = transform_teacher_dataframe(professores_raw)

print('Aulas tratadas:', df_result.shape)
print('Professores tratados:', df_professor.shape)
display(df_result['dias da semana'].value_counts().rename('qtd_turmas_por_dia'))

## 2. Dependencias do solver

Estas funcoes montam o modelo com a mesma ordem de restricoes usada em `TeacherScheduler.schedule_teachers()` e retornam o status real do CP-SAT.

In [ ]:
STATUS_MAP = {
    cp_model.OPTIMAL: 'OPTIMAL',
    cp_model.FEASIBLE: 'FEASIBLE',
    cp_model.INFEASIBLE: 'INFEASIBLE',
    cp_model.MODEL_INVALID: 'MODEL_INVALID',
    cp_model.UNKNOWN: 'UNKNOWN',
}


def status_nome(status):
    return STATUS_MAP.get(status, str(status))


def montar_modelo_com_mesma_logica(Ts, use_soft_constrait):
    Ts.create_model()
    Ts.create_variables()
    Ts.add_teacher_pre_alocation()
    Ts.add_teacher_constraints()
    Ts.add_schedule_constraints()
    Ts.add_unidade_constraints()
    Ts.add_impossible_group_constraints()
    Ts.add_consecutive_group_constraints()
    Ts.add_modalidades_constraints()
    Ts.add_grupo_constraints()
    Ts.add_estagio_constraints()
    Ts.add_online_constraints()
    Ts.add_time_constraints()
    Ts.add_intensive_constraints()
    Ts.add_restrictions_constraints()
    Ts.teacher_at_least_1_class_constraints()

    if use_soft_constrait == 0:
        Ts.add_class_per_teacher_constraints_hard()
        Ts.add_consecutive_teacher_constraints_soft()
    elif use_soft_constrait == 1:
        Ts.add_class_per_teacher_constraints_weighted()
        Ts.add_consecutive_teacher_constraints()
    elif use_soft_constrait == 2:
        Ts.add_class_per_teacher_constraints_double_weighted()

    return Ts


def resolver_modelo(Ts, seed=42, max_time_in_seconds=20):
    solver = cp_model.CpSolver()
    solver.parameters.random_seed = seed
    solver.parameters.max_time_in_seconds = max_time_in_seconds
    solver.parameters.enumerate_all_solutions = False

    inicio = time.time()
    status = solver.Solve(Ts.model)
    duracao = round(time.time() - inicio, 2)

    alocacoes = []
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        for grupo in Ts.df_class['nome grupo'].unique():
            for professor in Ts.df_teach['TEACHER'].unique():
                if solver.Value(Ts.alocacoes[(professor, grupo)]):
                    alocacoes.append({
                        'professores_alocados': professor,
                        'nome grupo': grupo,
                    })

    return {
        'status_codigo': status,
        'status': status_nome(status),
        'tempo_s': duracao,
        'qtd_alocacoes': len(alocacoes),
        'alocacoes': pd.DataFrame(alocacoes),
    }

## 3. Comparar `use_soft_constrait` por dia

Use esta funcao para ver se um dia funciona com `use_soft_constrait=3`, mas quebra com `0`, `1` ou `2`.

In [ ]:
def testar_use_soft_por_dia(
    df_class,
    df_teach,
    dia,
    opcoes=(0, 1, 2, 3),
    seed=42,
    max_time_in_seconds=20,
):
    df_dia = df_class[df_class['dias da semana'] == dia].reset_index(drop=True)
    resultados = []
    detalhes = {}

    if df_dia.empty:
        raise ValueError(f'Nenhuma turma encontrada para o dia: {dia}')

    for use_soft in opcoes:
        try:
            Ts = TeacherScheduler(df_dia, df_teach)
            montar_modelo_com_mesma_logica(Ts, use_soft)
            resultado = resolver_modelo(
                Ts,
                seed=seed,
                max_time_in_seconds=max_time_in_seconds,
            )
            detalhes[use_soft] = resultado
            resultados.append({
                'dia': dia,
                'use_soft_constrait': use_soft,
                'status': resultado['status'],
                'qtd_alocacoes': resultado['qtd_alocacoes'],
                'tempo_s': resultado['tempo_s'],
            })
        except Exception as exc:
            detalhes[use_soft] = {'erro': repr(exc)}
            resultados.append({
                'dia': dia,
                'use_soft_constrait': use_soft,
                'status': 'ERRO',
                'qtd_alocacoes': 0,
                'tempo_s': np.nan,
                'erro': repr(exc),
            })

    tabela = pd.DataFrame(resultados)
    display(tabela)
    return tabela, detalhes

## 4. Teste incremental das restricoes

Use esta funcao para descobrir exatamente em qual metodo o modelo passa a ficar `INFEASIBLE`.

In [ ]:
def passos_restricoes(use_soft_constrait):
    passos = [
        ('add_teacher_pre_alocation', lambda Ts: Ts.add_teacher_pre_alocation()),
        ('add_teacher_constraints', lambda Ts: Ts.add_teacher_constraints()),
        ('add_schedule_constraints', lambda Ts: Ts.add_schedule_constraints()),
        ('add_unidade_constraints', lambda Ts: Ts.add_unidade_constraints()),
        ('add_impossible_group_constraints', lambda Ts: Ts.add_impossible_group_constraints()),
        ('add_consecutive_group_constraints', lambda Ts: Ts.add_consecutive_group_constraints()),
        ('add_modalidades_constraints', lambda Ts: Ts.add_modalidades_constraints()),
        ('add_grupo_constraints', lambda Ts: Ts.add_grupo_constraints()),
        ('add_estagio_constraints', lambda Ts: Ts.add_estagio_constraints()),
        ('add_online_constraints', lambda Ts: Ts.add_online_constraints()),
        ('add_time_constraints', lambda Ts: Ts.add_time_constraints()),
        ('add_intensive_constraints', lambda Ts: Ts.add_intensive_constraints()),
        ('add_restrictions_constraints', lambda Ts: Ts.add_restrictions_constraints()),
        ('teacher_at_least_1_class_constraints', lambda Ts: Ts.teacher_at_least_1_class_constraints()),
    ]

    if use_soft_constrait == 0:
        passos.extend([
            ('add_class_per_teacher_constraints_hard', lambda Ts: Ts.add_class_per_teacher_constraints_hard()),
            ('add_consecutive_teacher_constraints_soft', lambda Ts: Ts.add_consecutive_teacher_constraints_soft()),
        ])
    elif use_soft_constrait == 1:
        passos.extend([
            ('add_class_per_teacher_constraints_weighted', lambda Ts: Ts.add_class_per_teacher_constraints_weighted()),
            ('add_consecutive_teacher_constraints', lambda Ts: Ts.add_consecutive_teacher_constraints()),
        ])
    elif use_soft_constrait == 2:
        passos.append(
            ('add_class_per_teacher_constraints_double_weighted', lambda Ts: Ts.add_class_per_teacher_constraints_double_weighted())
        )

    return passos


def testar_restricoes_incrementais(
    df_class,
    df_teach,
    dia,
    use_soft_constrait=3,
    seed=42,
    max_time_in_seconds=10,
    parar_no_inviavel=False,
):
    df_dia = df_class[df_class['dias da semana'] == dia].reset_index(drop=True)

    if df_dia.empty:
        raise ValueError(f'Nenhuma turma encontrada para o dia: {dia}')

    Ts = TeacherScheduler(df_dia, df_teach)
    Ts.create_model()
    Ts.create_variables()

    linhas = []
    for nome_passo, aplicar in passos_restricoes(use_soft_constrait):
        try:
            aplicar(Ts)
            resultado = resolver_modelo(
                Ts,
                seed=seed,
                max_time_in_seconds=max_time_in_seconds,
            )
            linhas.append({
                'dia': dia,
                'use_soft_constrait': use_soft_constrait,
                'passo': nome_passo,
                'status': resultado['status'],
                'qtd_alocacoes': resultado['qtd_alocacoes'],
                'tempo_s': resultado['tempo_s'],
            })

            if parar_no_inviavel and resultado['status'] in ('INFEASIBLE', 'MODEL_INVALID'):
                break
        except Exception as exc:
            linhas.append({
                'dia': dia,
                'use_soft_constrait': use_soft_constrait,
                'passo': nome_passo,
                'status': 'ERRO',
                'qtd_alocacoes': 0,
                'tempo_s': np.nan,
                'erro': repr(exc),
            })
            break

    tabela = pd.DataFrame(linhas)
    display(tabela)
    return tabela


def comparar_restricoes_incrementais(
    df_class,
    df_teach,
    dia,
    opcoes=(0, 1, 2, 3),
    seed=42,
    max_time_in_seconds=10,
    parar_no_inviavel=False,
):
    tabelas = []
    for use_soft in opcoes:
        print(f'Testando {dia} com use_soft_constrait={use_soft}')
        tabela = testar_restricoes_incrementais(
            df_class,
            df_teach,
            dia,
            use_soft_constrait=use_soft,
            seed=seed,
            max_time_in_seconds=max_time_in_seconds,
            parar_no_inviavel=parar_no_inviavel,
        )
        tabelas.append(tabela)

    resultado = pd.concat(tabelas, ignore_index=True)
    display(resultado)
    return resultado

## 5. Exemplos de uso

Rode as celulas abaixo para investigar `TERÇA`, `QUINTA` ou outro dia.

In [ ]:
comparacao_terca, detalhes_terca = testar_use_soft_por_dia(
    df_result,
    df_professor,
    'TERÇA',
    opcoes=(0, 1, 2, 3),
    seed=42,
)

comparacao_quinta, detalhes_quinta = testar_use_soft_por_dia(
    df_result,
    df_professor,
    'QUINTA',
    opcoes=(0, 1, 2, 3),
    seed=42,
)

In [ ]:
incremental_quinta = comparar_restricoes_incrementais(
    df_result,
    df_professor,
    'QUINTA',
    opcoes=(0, 1, 2, 3),
    seed=42,
    parar_no_inviavel=True,
)